In [20]:
import boto3
import json
import uuid
import base64

In [21]:
client = boto3.client("bedrock-agentcore", region_name="us-east-1")

<h2>Basic image agent test</h2>

In [22]:
# --- Point this at your image file ---
IMAGE_PATH = "11.png"  # change to your image path
IMAGE_FORMAT = "png"           # one of: png, jpeg, gif, webp

# Read and base64-encode the image
with open(IMAGE_PATH, "rb") as f:
    image_b64 = base64.b64encode(f.read()).decode("utf-8")

# Build the payload
image_payload = json.dumps({
    "prompt": "Describe what you see in this image in detail.",
    "image_base64": image_b64,
    "image_format": IMAGE_FORMAT,
})

# Generate a fresh session for the image request
image_session_id = str(uuid.uuid4())

response = client.invoke_agent_runtime(
    agentRuntimeArn='arn:aws:bedrock-agentcore:us-east-1:014498646416:runtime/tom_hackathon-ChO02O61W2',
    runtimeSessionId=image_session_id,
    payload=image_payload,
)

response_body = response['response'].read()
response_data = json.loads(response_body)
print("Agent Response:", response_data)

Agent Response: {'result': {'role': 'assistant', 'content': [{'text': '## Image Description\n\nThis is a **men\'s fashion advertisement** featuring:\n\n### Model & Outfit\n- A **smiling young man** with short dark hair, sitting on a stool\n- He\'s wearing:\n  - A **dark gray/charcoal jacket** (outerwear) with chest pockets\n  - A **dark red/burgundy shirt** underneath\n  - **Bright orange pants**\n  - A **brown leather messenger bag** worn across his body\n  - A **brown belt**\n\n### Background\n- A **dark gray, textured backdrop** giving a studio feel\n\n### Text/Promotional Content\n- **Large headline:** "STAY WARM IN STYLE"\n- **Subheading:** "shop this season\'s outerwear for men."\n- **Promotional offer:** "UP TO 40% OFF MEN\'S OUTERWEAR FROM LEVI\'S®, TOMMY HILFIGER®, GUESS, NAUTICA®, CALVIN KLEIN AND MORE!"\n- **Call-to-action button:** "SHOP ALL OUTERWEAR ▶"\n\n### Overall Impression\nThis appears to be a **retail/department store advertisement** promoting a seasonal sale on me

<h2>Ranking agent tests</h2>

In [23]:
# Replace with your ranking agent's ARN after deploying to AgentCore
RANKING_AGENT_ARN = "arn:aws:bedrock-agentcore:us-east-1:014498646416:runtime/ranking_agent-0XoBDAHnnV"

# Example user IAB preference profile (from Stage 1 LR or ground truth)
user_profile = {
    "Automotive": 4.2,
    "Sports": 3.8,
    "Food & Drink": 3.1,
    "Style & Fashion": 2.9,
    "Technology & Computing": 1.1,
    "Health & Fitness": 2.5,
    "Travel": 1.8,
    "Home & Garden": 0.4,
    "Business": 0.2,
    "Education": 0.1,
}

# Example candidate ads with their IAB profiles
candidate_ads = [
    {
        "id": "Cat3_7",
        "iab_profile": {"Sports": 1, "Health & Fitness": 1},
    },
    {
        "id": "Cat5_2",
        "iab_profile": {"Food & Drink": 1, "Health & Fitness": 1},
    },
    {
        "id": "Cat1_11",
        "iab_profile": {"Automotive": 1, "Technology & Computing": 1},
    },
    {
        "id": "Cat8_4",
        "iab_profile": {"Style & Fashion": 1, "Sports": 1},
    },
    {
        "id": "Cat12_9",
        "iab_profile": {"Home & Garden": 1, "Style & Fashion": 1},
    },
]

ranking_payload = json.dumps({
    "user_profile": user_profile,
    "candidate_ads": candidate_ads,
    "top_k": 5,
})

session_id = str(uuid.uuid4())

response = client.invoke_agent_runtime(
    agentRuntimeArn=RANKING_AGENT_ARN,
    runtimeSessionId=session_id,
    payload=ranking_payload,
)

response_body = response["response"].read()
response_data = json.loads(response_body)

# Pretty-print the ranked results
result = response_data.get("result", response_data)
print(json.dumps(result, indent=2))

{
  "ranked_ads": [
    {
      "rank": 1,
      "id": "Cat8_4",
      "score": 6.7,
      "reasoning": "This ad sits at the intersection of Style & Fashion (2.9) and Sports (3.8), two of the user's strongest affinity categories, creating a powerful athleisure synergy. Visually, expect dynamic action imagery paired with aspirational lifestyle aesthetics \u2014 bold colors, athletic silhouettes, and performance-meets-style branding \u2014 all of which are proven engagement drivers for users who value both performance and appearance. The cross-category overlap means the ad appeals to the user's identity on multiple dimensions simultaneously."
    },
    {
      "rank": 2,
      "id": "Cat3_7",
      "score": 6.3,
      "reasoning": "Combining Sports (3.8) and Health & Fitness (2.5), this ad aligns strongly with the user's active lifestyle orientation. High-energy action shots, motivational messaging, and competitive imagery are likely visual hallmarks that will resonate deeply given the 

<h2>Alternative Feature Extraction Tests</h2>

In [24]:
# Replace with your feature extraction agent's ARN after deploying to AgentCore
FEATURE_AGENT_ARN = "arn:aws:bedrock-agentcore:us-east-1:014498646416:runtime/extract_alternative_metadata-TYFhVABA78"

IMAGE_PATH = "11.png"
IMAGE_FORMAT = "png"

with open(IMAGE_PATH, "rb") as f:
    image_b64 = base64.b64encode(f.read()).decode("utf-8")

extraction_payload = json.dumps({
    "image_base64": image_b64,
    "image_format": IMAGE_FORMAT,
})

session_id = str(uuid.uuid4())

response = client.invoke_agent_runtime(
    agentRuntimeArn=FEATURE_AGENT_ARN,
    runtimeSessionId=session_id,
    payload=extraction_payload,
)

response_body = response["response"].read()
response_data = json.loads(response_body)

result = response_data.get("result", response_data)
print(json.dumps(result, indent=2))

{
  "aesthetic_score": 7,
  "visual_clutter": 3,
  "focal_point_presence": 8,
  "contrast_level": 7,
  "visual_saliency_score": 8,
  "primary_subject_type": "person",
  "product_visibility": 8,
  "usage_context": "lifestyle",
  "brand_prominence": 5,
  "logo_present": false,
  "word_count": 31,
  "text_density": 5,
  "readability": 8,
  "value_proposition_present": true,
  "cta_present": true,
  "offer_present": true,
  "emotion_valence": "positive",
  "human_presence": true,
  "perceived_credibility": 7,
  "spamminess_score": 3
}


## Full Pipeline Demo: Classify Images → Rank for User → ImagePrediction Output

This section demonstrates the end-to-end agent pipeline that the team can later
copy into `app/services/model_service.py` (`CustomInferenceInterface.predict`).

In [26]:
import sys, os
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

# Add project root so we can import src modules
PROJECT_ROOT = Path(os.getcwd()).parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader.agent_processing.categories_t1 import (
    load_categories,
    build_categorization_prompt,
)

IMAGE_AGENT_ARN = "arn:aws:bedrock-agentcore:us-east-1:014498646416:runtime/tom_hackathon-ChO02O61W2"
RANKING_AGENT_ARN = "arn:aws:bedrock-agentcore:us-east-1:014498646416:runtime/ranking_agent-0XoBDAHnnV"

IAB_CATEGORIES = load_categories()
IAB_PROMPT = build_categorization_prompt()

print(f"Loaded {len(IAB_CATEGORIES)} IAB categories")
print(f"Prompt preview: {IAB_PROMPT[:120]}...")

Loaded 34 IAB categories
Prompt preview: Assign IAB tier1 categories to the image. Return a list of categories separated by commas. Do not provide any reasoning ...


In [27]:
# --- Step 1: Classify images into IAB categories (parallel) ---

def detect_image_format(image_bytes: bytes) -> str:
    """Sniff format from magic bytes (replicates batch_invoke_ads logic)."""
    if image_bytes.startswith(b"\x89PNG\r\n\x1a\n"):
        return "png"
    if image_bytes.startswith(b"\xff\xd8\xff"):
        return "jpeg"
    if image_bytes[:6] in (b"GIF87a", b"GIF89a"):
        return "gif"
    if image_bytes[:4] == b"RIFF" and image_bytes[8:12] == b"WEBP":
        return "webp"
    raise ValueError(f"Unsupported format (magic: {image_bytes[:16]!r})")


def classify_image(slot_index, filename, image_bytes):
    """Call basic image agent with IAB prompt, return parsed iab_profile dict."""
    fmt = detect_image_format(image_bytes)
    image_b64 = base64.b64encode(image_bytes).decode("utf-8")

    payload = json.dumps({
        "prompt": IAB_PROMPT,
        "image_base64": image_b64,
        "image_format": fmt,
    })

    resp = client.invoke_agent_runtime(
        agentRuntimeArn=IMAGE_AGENT_ARN,
        runtimeSessionId=str(uuid.uuid4()),
        payload=payload,
    )
    body = json.loads(resp["response"].read().decode("utf-8"))
    text = body["result"]["content"][0]["text"]

    # Parse comma-separated categories into binary iab_profile
    returned_cats = [c.strip() for c in text.split(",")]
    valid_set = set(IAB_CATEGORIES)
    iab_profile = {cat: 1 for cat in returned_cats if cat in valid_set}

    print(f"  slot_{slot_index} ({filename}): {list(iab_profile.keys())}")
    return slot_index, filename, iab_profile


# Load test images — adjust paths as needed
test_images = [
    ("11.png", Path("11.png")),
    # Add more images here:
    # ("another.png", Path("path/to/another.png")),
]

image_payloads = []
for name, path in test_images:
    image_payloads.append((name, path.read_bytes()))

print(f"Classifying {len(image_payloads)} image(s) into IAB categories...\n")

classified = {}
with ThreadPoolExecutor(max_workers=min(8, len(image_payloads))) as pool:
    futures = {
        pool.submit(classify_image, i, name, blob): i
        for i, (name, blob) in enumerate(image_payloads)
    }
    for fut in as_completed(futures):
        slot_index, filename, iab_profile = fut.result()
        classified[slot_index] = {
            "filename": filename,
            "iab_profile": iab_profile,
        }

print(f"\nClassified {len(classified)} image(s).")

Classifying 1 image(s) into IAB categories...

  slot_0 (11.png): ['Style & Fashion', 'Shopping']

Classified 1 image(s).


In [28]:
# --- Step 2: Build inputs and call the ranking agent ---

# Placeholder user IAB profile — uniform scores for now.
# Replace with LR-predicted profile later.
user_iab_profile = {cat: 1.0 for cat in IAB_CATEGORIES}

# Build candidate_ads from classification results
candidate_ads = [
    {"id": f"slot_{i}", "iab_profile": classified[i]["iab_profile"]}
    for i in sorted(classified.keys())
]

print("User IAB profile: uniform (all categories = 1.0)")
print(f"Candidate ads: {len(candidate_ads)}")
for ad in candidate_ads:
    cats = [c for c, v in ad["iab_profile"].items() if v]
    print(f"  {ad['id']}: {cats}")

# Call the ranking agent
ranking_payload = json.dumps({
    "user_profile": user_iab_profile,
    "candidate_ads": candidate_ads,
    "top_k": len(candidate_ads),
})

print(f"\nCalling ranking agent...")
resp = client.invoke_agent_runtime(
    agentRuntimeArn=RANKING_AGENT_ARN,
    runtimeSessionId=str(uuid.uuid4()),
    payload=ranking_payload,
)

ranking_response = json.loads(resp["response"].read().decode("utf-8"))
ranking_result = ranking_response.get("result", ranking_response)

print("\nRanking agent response:")
print(json.dumps(ranking_result, indent=2))

User IAB profile: uniform (all categories = 1.0)
Candidate ads: 1
  slot_0: ['Style & Fashion', 'Shopping']

Calling ranking agent...

Ranking agent response:
{
  "ranked_ads": [
    {
      "rank": 1,
      "id": "slot_0",
      "score": 2.0,
      "reasoning": "This ad aligns with the user's equal-weight interest in Style & Fashion and Shopping, both scoring 1.0 in the profile. Since all categories carry identical affinity scores, this ad's two-category overlap yields the maximum possible dot-product score of 2.0 among the candidates. Visual features typical of fashion/shopping ads \u2014 such as aspirational lifestyle imagery, product close-ups, vibrant color palettes, and trend-forward styling \u2014 are likely to drive engagement with this broadly interested user."
    }
  ],
  "analysis": "This user has a perfectly uniform IAB preference profile, meaning no single category dominates and any ad is evaluated purely on how many relevant categories it covers. With only one candidate 

In [29]:
# --- Step 3: Map ranking response to ImagePrediction shape ---
#
# This is the exact output contract that app/services/model_service.py expects.
# The team can copy this mapping into CustomInferenceInterface.predict.

ranked_ads = ranking_result.get("ranked_ads", [])

# Build a lookup from ad id back to slot index
id_to_slot = {f"slot_{i}": i for i in classified.keys()}

predictions = []
for ad in ranked_ads:
    slot = id_to_slot[ad["id"]]
    info = classified[slot]
    iab_attrs = {cat: str(val) for cat, val in info["iab_profile"].items()}

    predictions.append({
        "slot_index": slot,
        "filename": info["filename"],
        "affinity": float(ad["score"]),
        "reason": ad["reasoning"],
        "image_attributes": iab_attrs,
    })

# Sort by affinity descending (matching what the app router expects)
predictions.sort(key=lambda p: p["affinity"], reverse=True)

print("=== ImagePrediction output (ready for model_service.py) ===\n")
for i, pred in enumerate(predictions, 1):
    print(f"Rank {i}:")
    print(f"  slot_index: {pred['slot_index']}")
    print(f"  filename:   {pred['filename']}")
    print(f"  affinity:   {pred['affinity']}")
    print(f"  reason:     {pred['reason'][:120]}...")
    print(f"  image_attributes: {pred['image_attributes']}")
    print()

if ranking_result.get("analysis"):
    print(f"Overall analysis: {ranking_result['analysis']}")

=== ImagePrediction output (ready for model_service.py) ===

Rank 1:
  slot_index: 0
  filename:   11.png
  affinity:   2.0
  reason:     This ad aligns with the user's equal-weight interest in Style & Fashion and Shopping, both scoring 1.0 in the profile. S...
  image_attributes: {'Style & Fashion': '1', 'Shopping': '1'}

Overall analysis: This user has a perfectly uniform IAB preference profile, meaning no single category dominates and any ad is evaluated purely on how many relevant categories it covers. With only one candidate ad available, slot_0's dual coverage of Style & Fashion and Shopping makes it the clear top pick. For this user, ads that combine multiple overlapping themes — such as fashion paired with travel, or shopping tied to personal celebrations — will consistently outperform single-category ads. Visually rich, multi-contextual creative (e.g., lifestyle scenes that blend fashion, home aesthetics, or social events) would be especially effective at capturing this user's